# **⏳ Flight Disruption & Delay Predictor**
- **Objective:** Classify whether an upcoming flight route will face a critical delay (greater than $15$ minutes) and predict the exact duration of the delay vector.
- **The Math & Architecture:** A dual-stage ensemble framework using **LightGBM** or **XGBoost**.
    - *Stage 1:* Binary classification estimating the probability of a delay ($P(\text{Delay}) \ge \theta$).
    - *Stage 2:* If a delay is triggered, a gradient-boosted regression tree estimates the exact duration in minutes.
        
Feature engineering is the heavy lifter here: cyclic encodings (using sine and cosine transformations) must be applied to time parameters like `day_of_week` and `scheduled_dep_hour` to capture temporal continuity.


This dataset contains detailed flight performance and delay information for domestic flights in **2024**, merged from monthly BTS TranStats files into a single cleaned dataset. It includes over **7 million rows and 35 columns**, providing comprehensive information on scheduled and actual flight times, delays, cancellations, diversions, and distances between airports. The dataset is suitable for exploratory data analysis (EDA), machine learning tasks such as delay prediction, time series analysis, and airline/airport performance studies.

## **Dataset Overview**
| Column Name | Description |
| --- | --- |
| `year` | Year of flight |
| `month` | Month of flight (1–12) |
| `day_of_month` | Day of the month |
| `day_of_week` | Day of week (1=Monday … 7=Sunday) |
| `fl_date` | Flight date (YYYY-MM-DD) |
| `op_unique_carrier` | Unique carrier code |
| `op_carrier_fl_num` | Flight number for reporting airline |
| `origin` | Origin airport code |
| `origin_city_name` | Origin city name |
| `origin_state_nm` | Origin state name |
| `dest` | Destination airport code |
| `dest_city_name` | Destination city name |
| `dest_state_nm` | Destination state name |
| `crs_dep_time` | Scheduled departure time (local, hhmm) |
| `dep_time` | Actual departure time (local, hhmm) |
| `dep_delay` | Departure delay in minutes (negative if early) |
| `taxi_out` | Taxi out time in minutes |
| `wheels_off` | Wheels-off time (local, hhmm) |
| `wheels_on` | Wheels-on time (local, hhmm) |
| `taxi_in` | Taxi in time in minutes |
| `crs_arr_time` | Scheduled arrival time (local, hhmm) |
| `arr_time` | Actual arrival time (local, hhmm) |
| `arr_delay` | Arrival delay in minutes (negative if early) |
| `cancelled` | Cancelled flight indicator (0=No, 1=Yes) |
| `cancellation_code` | Reason for cancellation (if cancelled) |
| `diverted` | Diverted flight indicator (0=No, 1=Yes) |
| `crs_elapsed_time` | Scheduled elapsed time in minutes |
| `actual_elapsed_time` | Actual elapsed time in minutes |
| `air_time` | Flight time in minutes |
| `distance` | Distance between origin and destination (miles) |
| `carrier_delay` | Carrier-related delay in minutes |
| `weather_delay` | Weather-related delay in minutes |
| `nas_delay` | National Air System delay in minutes |
| `security_delay` | Security delay in minutes |
| `late_aircraft_delay` | Late aircraft delay in minutes |

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
from matplotlib import pyplot as plt

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/hrishitpatil/flight-data-2024/flight_data_2024_sample.csv
/kaggle/input/datasets/hrishitpatil/flight-data-2024/flight_data_2024_data_dictionary.csv
/kaggle/input/datasets/hrishitpatil/flight-data-2024/flight_data_2024.csv


In [2]:
data = pd.read_csv('/kaggle/input/datasets/hrishitpatil/flight-data-2024/flight_data_2024_sample.csv')
data.shape

(10000, 35)

In [3]:
data.isnull().sum()

year                      0
month                     0
day_of_month              0
day_of_week               0
fl_date                   0
op_unique_carrier         0
op_carrier_fl_num         0
origin                    0
origin_city_name          0
origin_state_nm           0
dest                      0
dest_city_name            0
dest_state_nm             0
crs_dep_time              0
dep_time                116
dep_delay               116
taxi_out                120
wheels_off              120
wheels_on               127
taxi_in                 127
crs_arr_time              0
arr_time                127
arr_delay               164
cancelled                 0
cancellation_code      9878
diverted                  0
crs_elapsed_time          0
actual_elapsed_time     164
air_time                164
distance                  0
carrier_delay             0
weather_delay             0
nas_delay                 0
security_delay            0
late_aircraft_delay       0
dtype: int64